# Penobscot River Basin, Maine — Comprehensive Watershed Report

A complete hydrologic analysis of the Penobscot River basin (HUC4 0102),
Maine's largest watershed. The Penobscot drains 8,570 sq mi of central
Maine — from the slopes of Mount Katahdin to Penobscot Bay — carrying
snowmelt, spring rain, and the memory of New England's boreal forests
through one of the last great Atlantic salmon rivers in the United States.

**Rivers:** Penobscot River at West Enfield (100+ yr record),
Piscataquis River near Dover-Foxcroft, Mattawamkeag River near Mattawamkeag

**Data:** USGS NWIS daily-value discharge (federal public domain),
ONI (ENSO) and PDO climate indices.

**HUC4:** 0102 (Penobscot)

In [ ]:
# ── imports & paths ──────────────────────────────────────────────────────
import sys
from pathlib import Path

REPO = Path(".").resolve().parent
sys.path.insert(0, str(REPO))

import numpy as np
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.ticker as ticker

from src import flow_metrics as fm
from tools.nwis_gauge import GaugeProvider
from tools.climate_index import ClimateIndexProvider
from tools.monthly_flow import MONTH_ABBR

NAS_ROOT = "/Volumes/home/data/hydro-art"
# Fall back to local cache if NAS unmounted
if not Path(NAS_ROOT).exists():
    NAS_ROOT = str(REPO / "cache")
    Path(NAS_ROOT).mkdir(exist_ok=True)

%matplotlib inline
plt.rcParams.update({
    "figure.facecolor": "#07080c",
    "axes.facecolor": "#10121b",
    "axes.edgecolor": "#232838",
    "axes.labelcolor": "#e6ebf5",
    "text.color": "#e6ebf5",
    "xtick.color": "#8891a8",
    "ytick.color": "#8891a8",
    "figure.dpi": 140,
    "savefig.dpi": 140,
    "font.size": 10,
})

# neon palette
CYAN = "#00ffff"
MAGENTA = "#ff4d9a"
LIME = "#00ff9c"
VIOLET = "#9d00ff"
AMBER = "#ff9c3a"
RED = "#ff4d5e"
COOL_BLUE = "#00e5ff"
MUTED = "#8891a8"
LIGHT = "#e6ebf5"

FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

def complete_years(obs):
    """Filter to years where all 12 months are finite (no NaN gaps)."""
    return {y: v for y, v in obs.items() if np.all(np.isfinite(v))}

def save(fig, name):
    fig.savefig(FIGURES / f"penobscot_{name}.png", bbox_inches="tight", pad_inches=0.3)

print("hydro-art Penobscot Basin report — ready")

## 1. Data Acquisition

Three USGS gauges span the Penobscot basin, chosen for record length and
spatial coverage:

| Gauge | Site | Drainage (mi²) | Record |
|-------|------|-----------------|--------|
| Penobscot R. at West Enfield | 01034500 | 7,764 | 1902–present |
| Piscataquis R. near Dover-Foxcroft | 01031500 | 298 | 1902–present |
| Mattawamkeag R. near Mattawamkeag | 01030500 | 1,418 | 1934–present |

In [ ]:
# ── gauge data ────────────────────────────────────────────────────────────
penobscot = GaugeProvider("01034500", NAS_ROOT)
penobscot_obs = penobscot.monthly_means(1902, 2024)
print(f"Penobscot River at West Enfield: {penobscot.name}")
print(f"  {len(penobscot_obs)} years: {min(penobscot_obs)}–{max(penobscot_obs)}")

piscataquis = GaugeProvider("01031500", NAS_ROOT)
piscataquis_obs = piscataquis.monthly_means(1902, 2024)
print(f"\nPiscataquis River: {piscataquis.name}")
print(f"  {len(piscataquis_obs)} years: {min(piscataquis_obs)}–{max(piscataquis_obs)}")

mattawamkeag = GaugeProvider("01030500", NAS_ROOT)
mattawamkeag_obs = mattawamkeag.monthly_means(1934, 2024)
print(f"\nMattawamkeag River: {mattawamkeag.name}")
print(f"  {len(mattawamkeag_obs)} years: {min(mattawamkeag_obs)}–{max(mattawamkeag_obs)}")

# ── climate indices ────────────────────────────────────────────────────────
oni_prov = ClimateIndexProvider("oni", NAS_ROOT)
pdo_prov = ClimateIndexProvider("pdo", NAS_ROOT)

oni = oni_prov.index_by_year(1900, 2024)
pdo = pdo_prov.index_by_year(1900, 2024)

print(f"\nONI (ENSO): {len(oni)} years")
print(f"PDO: {len(pdo)} years")

# Complete-year filtered copies
pen_c = complete_years(penobscot_obs)
pisc_c = complete_years(piscataquis_obs)
matt_c = complete_years(mattawamkeag_obs)
print(f"\nComplete years — Penobscot: {len(pen_c)}, Piscataquis: {len(pisc_c)}, Mattawamkeag: {len(matt_c)}")

In [ ]:
# ── quick data summary table ──────────────────────────────────────────────
for name, obs in [("Penobscot", pen_c),
                   ("Piscataquis", pisc_c),
                   ("Mattawamkeag", matt_c)]:
    years = sorted(obs.keys())
    annual = [float(np.nanmean(obs[y])) for y in years]
    print(f"{name:20s}  {years[0]}–{years[-1]}  "
          f"{len(years):>3d} yrs  "
          f"mean {np.mean(annual):>8,.0f} cfs  "
          f"peak {max(annual):>8,.0f} cfs")

---
## 2. Year-over-Year Hydrographs

Every year's 12-month observed hydrograph overlaid — the foundational view.
Color = year (viridis palette, old → new). The Penobscot's snowmelt-dominated
regime is immediately visible: a sharp April–May peak as the northern Maine
snowpack melts, followed by low summer baseflow.

In [ ]:
def plot_hydrographs(obs, title, ax):
    years = sorted(obs.keys())
    cmap = plt.get_cmap("viridis")
    span = max(years) - min(years) or 1
    for y in years:
        c = cmap((y - min(years)) / span)
        ax.plot(range(1, 13), obs[y], color=c, lw=0.8, alpha=0.7)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_ABBR, fontsize=7)
    ax.set_ylabel("flow (cfs)")
    ax.set_title(title)
    sm = plt.cm.ScalarMappable(cmap="viridis",
        norm=plt.Normalize(vmin=min(years), vmax=max(years)))
    ax.figure.colorbar(sm, ax=ax, label="year", pad=0.01)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_hydrographs(pen_c, f"Penobscot ({min(pen_c)}–{max(pen_c)})", axes[0])
plot_hydrographs(pisc_c, f"Piscataquis ({min(pisc_c)}–{max(pisc_c)})", axes[1])
plot_hydrographs(matt_c, f"Mattawamkeag ({min(matt_c)}–{max(matt_c)})", axes[2])
plt.tight_layout()
save(fig, "hydrographs")
plt.show()

---
## 3. Typical Year — Mean Monthly Hydrograph + Confidence Band

The average seasonal shape with P10–P90 spread across years. The Penobscot
basin shows a classic boreal snowmelt regime: winter accumulation, explosive
spring freshet (April), rapid recession through May–June, and sustained low
flow from July through September. The P10–P90 band reveals how much the
spring peak varies year to year — driven by snowpack depth, melt timing,
and coincident rainfall.

In [ ]:
def plot_typical_year(obs, title, ax):
    years = sorted(obs.keys())
    stack = np.array([obs[y] for y in years])
    mean = np.nanmean(stack, axis=0)
    p10 = np.nanpercentile(stack, 10, axis=0)
    p25 = np.nanpercentile(stack, 25, axis=0)
    p75 = np.nanpercentile(stack, 75, axis=0)
    p90 = np.nanpercentile(stack, 90, axis=0)
    m = range(1, 13)
    ax.fill_between(m, p10, p90, color=CYAN, alpha=0.08, label="P10–P90")
    ax.fill_between(m, p25, p75, color=CYAN, alpha=0.15, label="P25–P75")
    ax.plot(m, mean, color=CYAN, lw=2, label="mean")
    cot = fm.center_of_timing(mean)
    ax.axvline(cot, color=AMBER, ls="--", lw=1, label=f"center = {cot:.1f}")
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_ABBR, fontsize=7)
    ax.set_ylabel("flow (cfs)")
    ax.set_title(title)
    ax.legend(fontsize=7)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_typical_year(pen_c, "Penobscot — Typical Year", axes[0])
plot_typical_year(pisc_c, "Piscataquis — Typical Year", axes[1])
plot_typical_year(matt_c, "Mattawamkeag — Typical Year", axes[2])
plt.tight_layout()
save(fig, "typical_year")
plt.show()

---
## 4. Long-Record Trend — Annual Mean + Mann-Kendall / Sen's Slope

Is flow increasing or decreasing over the gauge record? Non-parametric
Mann-Kendall trend test with Sen's robust slope estimator. For a 100+ year
record like the Penobscot, this captures century-scale shifts in precipitation,
land use, and dam operations.

In [ ]:
def plot_long_record(obs, title, ax):
    years = sorted(obs.keys())
    annual = np.array([float(np.nanmean(obs[y])) for y in years])
    mk = fm.mann_kendall(annual)
    slope = fm.sens_slope(annual)
    x = np.arange(len(years))
    fit = np.median(annual) + slope * (x - np.median(x))
    ax.plot(years, annual, "o-", color=CYAN, lw=1.2, ms=3, label="annual mean")
    ax.plot(years, fit, "--", color=VIOLET, lw=1.5,
            label=f"Sen {slope:+.2f} cfs/yr")
    # 10-year rolling mean
    if len(years) >= 10:
        rm = np.convolve(annual, np.ones(10)/10, mode="valid")
        ax.plot(years[4:-5], rm, color=AMBER, lw=1.5, alpha=0.8, label="10-yr rolling")
    ax.set_ylabel("mean annual flow (cfs)")
    ax.set_title(f"{title}\nMK: {mk.trend} (τ={mk.tau:+.3f}, p={mk.p:.4f})")
    ax.legend(fontsize=7)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_long_record(pen_c, "Penobscot — Long Record", axes[0])
plot_long_record(pisc_c, "Piscataquis — Long Record", axes[1])
plot_long_record(matt_c, "Mattawamkeag — Long Record", axes[2])
plt.tight_layout()
save(fig, "long_record")
plt.show()

---
## 5. Summer Low-Flow Trend

Jun–Aug minimum — the drought-stress signal. Are summers getting drier?
In snowmelt-dominated basins like the Penobscot, summer low flow depends on
how much winter snowpack recharged groundwater — less snow or earlier melt
means lower summer baseflow.

In [ ]:
def plot_low_flow(obs, title, ax):
    years = sorted(obs.keys())
    low = np.array([float(np.nanmin(obs[y][5:8])) for y in years])
    mk = fm.mann_kendall(low)
    slope = fm.sens_slope(low)
    x = np.arange(len(years))
    fit = np.median(low) + slope * (x - np.median(x))
    ax.bar(years, low, color=COOL_BLUE, alpha=0.7, width=0.8)
    ax.plot(years, fit, "--", color=MAGENTA, lw=1.5,
            label=f"Sen {slope:+.2f} cfs/yr ({mk.trend})")
    ax.set_ylabel("summer low flow (cfs)")
    ax.set_title(title)
    ax.legend(fontsize=7)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_low_flow(pen_c, "Penobscot — Summer Low", axes[0])
plot_low_flow(pisc_c, "Piscataquis — Summer Low", axes[1])
plot_low_flow(matt_c, "Mattawamkeag — Summer Low", axes[2])
plt.tight_layout()
save(fig, "low_flow")
plt.show()

---
## 6. Peak Flow Trend

Annual maximum monthly flow — the flood signal. In the Penobscot basin,
peak flow is almost always the April snowmelt freshet. Any trend here
reflects changes in snowpack volume, melt rate, and rain-on-snow events.

In [ ]:
def plot_peak_flow(obs, title, ax):
    years = sorted(obs.keys())
    peak = np.array([float(np.nanmax(obs[y])) for y in years])
    mk = fm.mann_kendall(peak)
    slope = fm.sens_slope(peak)
    x = np.arange(len(years))
    fit = np.median(peak) + slope * (x - np.median(x))
    ax.plot(years, peak, "o-", color=MAGENTA, lw=1.2, ms=3, label="annual peak")
    ax.plot(years, fit, "--", color=CYAN, lw=1.5,
            label=f"Sen {slope:+.2f} cfs/yr ({mk.trend})")
    ax.set_ylabel("peak monthly flow (cfs)")
    ax.set_title(title)
    ax.legend(fontsize=7)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_peak_flow(pen_c, "Penobscot — Peak Flow", axes[0])
plot_peak_flow(pisc_c, "Piscataquis — Peak Flow", axes[1])
plot_peak_flow(matt_c, "Mattawamkeag — Peak Flow", axes[2])
plt.tight_layout()
save(fig, "peak_flow")
plt.show()

---
## 7. Timing Drift — Center of Timing per Year

Is the hydrograph's mass shifting earlier in the year? The center-of-timing
trend detects snowmelt-dominated rivers arriving earlier under warming.
This is one of the clearest climate-change fingerprints in northern New
England hydrology — warmer winters advance the melt pulse.

In [ ]:
def plot_timing_drift(obs, title, ax):
    clean = complete_years(obs)
    tt = fm.center_of_timing_trend(clean)
    years = list(tt.years)
    centers = np.array(tt.center_months)
    ax.plot(years, centers, "o-", color=COOL_BLUE, lw=1.2, ms=3, label="center of timing")
    x = np.arange(len(years))
    fit = np.median(centers) + tt.slope_months_per_year * (x - np.median(x))
    ax.plot(years, fit, "--", color=VIOLET, lw=1.5,
            label=f"Sen {tt.days_per_decade:+.1f} d/dec")
    ax.set_ylabel("center of timing (month)")
    ax.set_title(f"{title}\ndrift: {tt.days_per_decade:+.1f} days/decade ({tt.trend})")
    ax.legend(fontsize=7)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_timing_drift(pen_c, "Penobscot — Timing Drift", axes[0])
plot_timing_drift(pisc_c, "Piscataquis — Timing Drift", axes[1])
plot_timing_drift(matt_c, "Mattawamkeag — Timing Drift", axes[2])
plt.tight_layout()
save(fig, "timing_drift")
plt.show()

---
## 8. ENSO Teleconnection — Peak Flow vs. ONI

The Penobscot sits in northern New England — ENSO teleconnections are
weaker here than in the Pacific Northwest, but La Niña winters tend to
bring colder/snowier conditions to the Northeast, which can amplify the
spring freshet. El Niño winters are often milder with less snowpack.

In [ ]:
def plot_enso(obs, index_by_year, title, ax, *, index_name="ONI"):
    years = sorted(obs.keys())
    peak_by_year = {y: float(np.nanmax(obs[y])) for y in years}
    common = sorted(set(peak_by_year) & set(index_by_year))
    if len(common) < 10:
        ax.text(0.5, 0.5, "insufficient overlap", ha="center", va="center",
                transform=ax.transAxes, color=MUTED)
        return
    xs = [index_by_year[y] for y in common]
    ys = [peak_by_year[y] for y in common]
    r = fm.correlate(peak_by_year, index_by_year)
    colors = [RED if x > 0.5 else COOL_BLUE if x < -0.5 else MUTED for x in xs]
    ax.scatter(xs, ys, c=colors, s=18, alpha=0.8, edgecolors="none")
    z = np.polyfit(xs, ys, 1)
    xr = np.linspace(min(xs), max(xs), 50)
    ax.plot(xr, np.polyval(z, xr), "--", color=VIOLET, lw=1.2)
    ax.set_xlabel(index_name)
    ax.set_ylabel("peak flow (cfs)")
    ax.set_title(f"{title}\nr = {r:+.3f}")

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_enso(pen_c, oni, "Penobscot × ONI", axes[0])
plot_enso(pisc_c, oni, "Piscataquis × ONI", axes[1])
plot_enso(matt_c, oni, "Mattawamkeag × ONI", axes[2])
plt.tight_layout()
save(fig, "enso")
plt.show()

---
## 9. ENSO Composite Hydrographs

Average seasonal shape for warm (El Niño), neutral, and cool (La Niña) years.
Shows how ENSO modulates the entire hydrograph, not just peak flow.

In [ ]:
def plot_composites(obs, index_by_year, title, ax, *, index_name="ONI"):
    clean = complete_years(obs)
    pc = fm.composite_hydrographs(clean, index_by_year)
    m = range(1, 13)
    for label, vec, color in (("warm", pc.warm, RED),
                               ("neutral", pc.neutral, MUTED),
                               ("cool", pc.cool, COOL_BLUE)):
        if vec is not None:
            n_yrs = len(getattr(pc, label + "_years"))
            ax.plot(m, vec, "-o", ms=4, lw=1.8, color=color,
                    label=f"{label} ({n_yrs} yr)")
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_ABBR, fontsize=7)
    ax.set_ylabel("flow (cfs)")
    ax.set_title(title)
    ax.legend(fontsize=7)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_composites(pen_c, oni, "Penobscot — ENSO Composites", axes[0])
plot_composites(pisc_c, oni, "Piscataquis — ENSO Composites", axes[1])
plot_composites(matt_c, oni, "Mattawamkeag — ENSO Composites", axes[2])
plt.tight_layout()
save(fig, "enso_composites")
plt.show()

---
## 10. Flow-Duration Curves by Decade

How has the entire flow *distribution* shifted over time? Each curve shows
the percentage of time a flow level is equaled or exceeded within that decade.
Shifts at the high end reveal changing flood behavior; shifts at the low end
reveal changing drought behavior.

In [ ]:
def plot_decade_fdc(obs, title, ax):
    clean = complete_years(obs)
    quantiles = [0, 5, 10, 25, 50, 75, 90, 95, 100]
    decades = fm.decade_flow_duration(clean, quantiles)
    cmap = plt.get_cmap("viridis")
    span = max(1, len(decades) - 1)
    for i, d in enumerate(decades):
        ax.plot(d.quantiles, np.clip(d.flows, 1e-2, None), "-o", ms=4,
                color=cmap(i / span), lw=1.6, label=f"{d.decade}s")
    ax.set_yscale("log")
    ax.set_xlabel("exceedance quantile (%)")
    ax.set_ylabel("flow (cfs, log)")
    ax.set_title(title)
    ax.legend(fontsize=6, ncol=2, frameon=False)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_decade_fdc(pen_c, "Penobscot — FDC by Decade", axes[0])
plot_decade_fdc(pisc_c, "Piscataquis — FDC by Decade", axes[1])
plot_decade_fdc(matt_c, "Mattawamkeag — FDC by Decade", axes[2])
plt.tight_layout()
save(fig, "decade_fdc")
plt.show()

---
## 11. Analog Years — Most Similar Historical Shapes

Which historical years had the most similar seasonal shape to the most
recent year? Ranked by Pearson correlation of the 12-month vectors
(removes mean/scale — focuses on shape). Useful for forecasting and
understanding what climate conditions produce similar hydrographs.

In [ ]:
def plot_analog_years(obs, title, ax, *, n=8):
    clean = complete_years(obs)
    target = max(clean.keys())
    analogs = fm.analog_years(clean, target, n=n)
    labels = [str(a.year) for a in analogs]
    sims = [a.similarity for a in analogs]
    colors = [CYAN if s > 0.9 else COOL_BLUE if s > 0.7 else MUTED for s in sims]
    bars = ax.barh(labels, sims, color=colors)
    ax.invert_yaxis()
    ax.set_xlabel("shape similarity (Pearson r)")
    ax.set_title(f"{title} — years most like {target}")
    for bar, s in zip(bars, sims):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f"{s:.2f}", va="center", fontsize=7, color=LIGHT)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_analog_years(pen_c, "Penobscot — Analogs", axes[0])
plot_analog_years(pisc_c, "Piscataquis — Analogs", axes[1])
plot_analog_years(matt_c, "Mattawamkeag — Analogs", axes[2])
plt.tight_layout()
save(fig, "analog_years")
plt.show()

---
## 12. Drought / Flood Record Book

Top-5 driest summers and wettest years in the entire gauge record.
The Penobscot's worst droughts and biggest floods often coincide with
regional climate extremes — the 1947 drought, the 1936 double-flood year.

In [ ]:
def plot_record_book(obs, title, ax):
    clean = complete_years(obs)
    rb = fm.record_book(clean, n=5)
    lines = [f"  {'DROUGHT / FLOOD RECORD BOOK':^50}", ""]
    lines.append("  Driest summers (Jun–Aug minimum):")
    lines.append(f"  {'Rank':>4}  {'Year':>4}  {'Flow (cfs)':>12}  {'Percentile':>10}")
    lines.append("  " + "─" * 40)
    for e in rb.driest_summers:
        lines.append(f"  #{e.rank:<3}  {e.year}  {e.value:>12.1f}  p{e.percentile*100:>8.0f}")
    lines += ["", "  Wettest years (annual peak):"]
    lines.append(f"  {'Rank':>4}  {'Year':>4}  {'Flow (cfs)':>12}  {'Percentile':>10}")
    lines.append("  " + "─" * 40)
    for e in rb.wettest_years:
        lines.append(f"  #{e.rank:<3}  {e.year}  {e.value:>12.1f}  p{e.percentile*100:>8.0f}")
    ax.text(0.05, 0.95, "\n".join(lines), transform=ax.transAxes,
            fontfamily="monospace", fontsize=8, va="top", color=LIGHT)
    ax.set_axis_off()
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_record_book(pen_c, "Penobscot — Records", axes[0])
plot_record_book(pisc_c, "Piscataquis — Records", axes[1])
plot_record_book(matt_c, "Mattawamkeag — Records", axes[2])
plt.tight_layout()
save(fig, "record_book")
plt.show()

---
## 13. Seasonal Ratio — Wet/Dry Season Balance

Ratio of wet-season (Nov–Apr) mean flow to dry-season (May–Oct) mean flow.
A higher ratio means a more seasonally concentrated flow regime — typical
of snowmelt-dominated basins where spring delivers most of the annual water.

In [ ]:
def plot_seasonal_ratio(obs, title, ax):
    clean = complete_years(obs)
    years = sorted(clean.keys())
    ratios = [fm.seasonal_ratio(clean[y]) for y in years]
    mk = fm.mann_kendall(ratios)
    slope = fm.sens_slope(ratios)
    x = np.arange(len(years))
    fit = np.median(ratios) + slope * (x - np.median(x))
    ax.plot(years, ratios, "o-", color=AMBER, lw=1.2, ms=3, label="wet:dry ratio")
    ax.plot(years, fit, "--", color=VIOLET, lw=1.5,
            label=f"Sen {slope:+.3f}/yr")
    ax.set_ylabel("seasonal ratio")
    ax.set_title(f"{title}\nMK: {mk.trend} (τ={mk.tau:+.3f})")
    ax.legend(fontsize=7)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_seasonal_ratio(pen_c, "Penobscot — Seasonal Ratio", axes[0])
plot_seasonal_ratio(pisc_c, "Piscataquis — Seasonal Ratio", axes[1])
plot_seasonal_ratio(matt_c, "Mattawamkeag — Seasonal Ratio", axes[2])
plt.tight_layout()
save(fig, "seasonal_ratio")
plt.show()

---
## 14. Flashiness Index — Richards-Baker Variability

How "spiky" is each year's hydrograph? Higher flashiness = more extreme
month-to-month swings. Snowmelt basins like the Penobscot are naturally
flashy — a slow winter build followed by a rapid spring release.

In [ ]:
def plot_flashiness(obs, title, ax):
    clean = complete_years(obs)
    years = sorted(clean.keys())
    flash = [fm.flashiness(clean[y]) for y in years]
    mk = fm.mann_kendall(flash)
    slope = fm.sens_slope(flash)
    x = np.arange(len(years))
    fit = np.median(flash) + slope * (x - np.median(x))
    ax.bar(years, flash, color=MAGENTA, alpha=0.7, width=0.8)
    ax.plot(years, fit, "--", color=CYAN, lw=1.5,
            label=f"Sen {slope:+.4f}/yr ({mk.trend})")
    ax.set_ylabel("Richards-Baker index")
    ax.set_title(title)
    ax.legend(fontsize=7)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_flashiness(pen_c, "Penobscot — Flashiness", axes[0])
plot_flashiness(pisc_c, "Piscataquis — Flashiness", axes[1])
plot_flashiness(matt_c, "Mattawamkeag — Flashiness", axes[2])
plt.tight_layout()
save(fig, "flashiness")
plt.show()

---
## 15. Rolling 30-Year Climate Normals

Sliding 30-year mean of annual flow — shows the slow multi-decadal
climate signal distinct from year-to-year variability. The Penobscot's
century-plus record is long enough to see the full AMO/PDO cycle.

In [ ]:
def plot_rolling_normals(obs, title, ax):
    years = sorted(obs.keys())
    annual = [float(np.nanmean(obs[y])) for y in years]
    if len(years) < 30:
        ax.text(0.5, 0.5, "< 30 years", ha="center", va="center",
                transform=ax.transAxes, color=MUTED)
        return
    rm = np.convolve(annual, np.ones(30)/30, mode="valid")
    ax.plot(years, annual, color=MUTED, lw=0.5, alpha=0.4, label="annual")
    ax.plot(years[14:-15], rm, color=CYAN, lw=2, label="30-yr normal")
    ax.axhline(np.mean(annual), color=AMBER, ls="--", lw=1, label="full-record mean")
    ax.set_ylabel("mean annual flow (cfs)")
    ax.set_title(title)
    ax.legend(fontsize=7)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_rolling_normals(pen_c, "Penobscot — 30-yr Normals", axes[0])
plot_rolling_normals(pisc_c, "Piscataquis — 30-yr Normals", axes[1])
plot_rolling_normals(matt_c, "Mattawamkeag — 30-yr Normals", axes[2])
plt.tight_layout()
save(fig, "rolling_normals")
plt.show()

---
## 16. Monthly Flow Histograms

Distribution of flow values for each calendar month across all years.
Shows the natural variability within each month — April's histogram
should be wide and right-skewed (big snowmelt years pull the tail).

In [ ]:
def plot_monthly_histograms(obs, title):
    years = sorted(obs.keys())
    fig, axes = plt.subplots(3, 4, figsize=(16, 10))
    for m in range(12):
        ax = axes[m // 4, m % 4]
        vals = [obs[y][m] for y in years if np.isfinite(obs[y][m])]
        ax.hist(vals, bins=20, color=CYAN, alpha=0.7, edgecolor="#232838")
        ax.axvline(np.median(vals), color=MAGENTA, ls="--", lw=1.2)
        ax.set_title(MONTH_ABBR[m], fontsize=9)
        ax.tick_params(labelsize=7)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    return fig

fig = plot_monthly_histograms(pen_c, "Penobscot — Monthly Flow Distributions")
save(fig, "monthly_hist")
plt.show()

---
## 17. Annual Flow Histogram + Kernel Density

The distribution of annual-mean flows across the full record.
A normal-ish distribution indicates stable climate; heavy tails or
bimodality hint at regime shifts.

In [ ]:
def plot_annual_histogram(obs, title, ax):
    years = sorted(obs.keys())
    annual = np.array([float(np.nanmean(obs[y])) for y in years])
    ax.hist(annual, bins=20, color=CYAN, alpha=0.7, edgecolor="#232838",
            density=True, label="histogram")
    # simple KDE
    from scipy.stats import gaussian_kde
    kde = gaussian_kde(annual)
    xr = np.linspace(annual.min() * 0.8, annual.max() * 1.2, 200)
    ax.plot(xr, kde(xr), color=MAGENTA, lw=2, label="KDE")
    ax.axvline(np.median(annual), color=AMBER, ls="--", lw=1.2, label="median")
    ax.set_xlabel("mean annual flow (cfs)")
    ax.set_ylabel("density")
    ax.set_title(title)
    ax.legend(fontsize=7)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_annual_histogram(pen_c, "Penobscot — Annual Distribution", axes[0])
plot_annual_histogram(pisc_c, "Piscataquis — Annual Distribution", axes[1])
plot_annual_histogram(matt_c, "Mattawamkeag — Annual Distribution", axes[2])
plt.tight_layout()
save(fig, "annual_hist")
plt.show()

---
## 18. Heatmap — Month × Year Flow Matrix

Color-coded matrix of flow by month and year — the complete picture.
Hot spots reveal individual extreme months; cool bands reveal
drought years. The April column should glow hottest.

In [ ]:
def plot_heatmap(obs, title):
    years = sorted(obs.keys())
    mat = np.array([obs[y] for y in years])
    fig, ax = plt.subplots(figsize=(10, max(6, len(years) * 0.12)))
    im = ax.imshow(mat, aspect="auto", cmap="magma",
                   extent=[0.5, 12.5, years[-1] + 0.5, years[0] - 0.5])
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_ABBR, fontsize=8)
    ax.set_ylabel("year")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="flow (cfs)", pad=0.02)
    plt.tight_layout()
    return fig

fig = plot_heatmap(pen_c, "Penobscot — Month × Year Flow")
save(fig, "heatmap")
plt.show()

---
## 19. Box Plots — Monthly Flow Distributions

Box-and-whisker for each calendar month — median, IQR, and outliers.
Complements the histograms with a compact multi-river comparison.

In [ ]:
def plot_monthly_box(obs, title, ax):
    years = sorted(obs.keys())
    data = [[obs[y][m] for y in years if np.isfinite(obs[y][m])] for m in range(12)]
    bp = ax.boxplot(data, patch_artist=True, widths=0.6,
                    boxprops=dict(facecolor="#10121b", edgecolor=CYAN),
                    whiskerprops=dict(color=MUTED),
                    capprops=dict(color=MUTED),
                    medianprops=dict(color=MAGENTA, lw=1.5),
                    flierprops=dict(marker=".", markerfacecolor=MUTED, markersize=3))
    ax.set_xticklabels(MONTH_ABBR, fontsize=7)
    ax.set_ylabel("flow (cfs)")
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_monthly_box(pen_c, "Penobscot — Monthly Box", axes[0])
plot_monthly_box(pisc_c, "Piscataquis — Monthly Box", axes[1])
plot_monthly_box(matt_c, "Mattawamkeag — Monthly Box", axes[2])
plt.tight_layout()
save(fig, "monthly_box")
plt.show()

---
## 20. PDO Teleconnection — Peak Flow vs. PDO

The Pacific Decadal Oscillation modulates North Atlantic storm tracks
on multi-decadal timescales. Positive PDO can shift the jet stream
and alter precipitation patterns over New England.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_enso(pen_c, pdo, "Penobscot × PDO", axes[0], index_name="PDO")
plot_enso(pisc_c, pdo, "Piscataquis × PDO", axes[1], index_name="PDO")
plot_enso(matt_c, pdo, "Mattawamkeag × PDO", axes[2], index_name="PDO")
plt.tight_layout()
save(fig, "pdo")
plt.show()

---
## 21. Climate Index Time Series — ONI + PDO

The raw climate indices overlaid with the flow record for visual
inspection. Warm (El Niño / +PDO) phases shaded red, cool shaded blue.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8))
years_oni = sorted(oni.keys())
vals_oni = [oni[y] for y in years_oni]
colors_oni = [RED if v > 0.5 else COOL_BLUE if v < -0.5 else MUTED for v in vals_oni]
ax1.bar(years_oni, vals_oni, color=colors_oni, width=0.8, alpha=0.8)
ax1.axhline(0, color=MUTED, lw=0.5)
ax1.set_ylabel("ONI")
ax1.set_title("Oceanic Niño Index (ONI)")

years_pdo = sorted(pdo.keys())
vals_pdo = [pdo[y] for y in years_pdo]
colors_pdo = [RED if v > 0.5 else COOL_BLUE if v < -0.5 else MUTED for v in vals_pdo]
ax2.bar(years_pdo, vals_pdo, color=colors_pdo, width=0.8, alpha=0.8)
ax2.axhline(0, color=MUTED, lw=0.5)
ax2.set_ylabel("PDO")
ax2.set_title("Pacific Decadal Oscillation (PDO)")
plt.tight_layout()
save(fig, "climate_indices")
plt.show()

---
## 22. Decadal Shift Analysis

Mean hydrograph shape per decade overlaid against the all-years mean.
Shows how the seasonal distribution of flow has evolved — is the spring
peak shrinking decade by decade? Is winter flow increasing?

In [ ]:
def plot_decadal_shift(obs, title, ax):
    years = sorted(obs.keys())
    overall_mean = np.nanmean([obs[y] for y in years], axis=0)
    decades = {}
    for y in years:
        d = (y // 10) * 10
        decades.setdefault(d, []).append(obs[y])
    cmap = plt.get_cmap("viridis")
    dec_list = sorted(decades.keys())
    span = max(1, len(dec_list) - 1)
    m = range(1, 13)
    ax.plot(m, overall_mean, color=MUTED, lw=2.5, ls="--", label="all-years mean", alpha=0.6)
    for i, d in enumerate(dec_list):
        c = cmap(i / span)
        dmean = np.nanmean(decades[d], axis=0)
        ax.plot(m, dmean, color=c, lw=1.5, label=f"{d}s")
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_ABBR, fontsize=7)
    ax.set_ylabel("flow (cfs)")
    ax.set_title(title)
    ax.legend(fontsize=6, ncol=2)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_decadal_shift(pen_c, "Penobscot — Decadal Shift", axes[0])
plot_decadal_shift(pisc_c, "Piscataquis — Decadal Shift", axes[1])
plot_decadal_shift(matt_c, "Mattawamkeag — Decadal Shift", axes[2])
plt.tight_layout()
save(fig, "decadal_shift")
plt.show()

---
## 23. Anomaly Stripes — Annual Departure from Normal

"Warming stripes" for water — each vertical bar is one year, colored by
how much its annual mean departs from the long-term average. Blue = wetter
than normal, red = drier. Clusters of color reveal multi-year wet/dry spells.

In [ ]:
def plot_anomaly_stripes(obs, title, ax):
    years = sorted(obs.keys())
    annual = np.array([float(np.nanmean(obs[y])) for y in years])
    normal = np.mean(annual)
    anomalies = annual - normal
    vmax = max(abs(anomalies.min()), abs(anomalies.max()))
    cmap_div = LinearSegmentedColormap.from_list(
        "anomaly", [COOL_BLUE, "#10121b", RED])
    for i, (y, a) in enumerate(zip(years, anomalies)):
        color = cmap_div((a / vmax + 1) / 2)
        ax.bar(y, 1, color=color, width=1.0, bottom=0)
    ax.set_yticks([])
    ax.set_xlabel("year")
    ax.set_title(title)

fig, axes = plt.subplots(3, 1, figsize=(18, 8))
plot_anomaly_stripes(pen_c, "Penobscot — Flow Anomaly Stripes", axes[0])
plot_anomaly_stripes(pisc_c, "Piscataquis — Flow Anomaly Stripes", axes[1])
plot_anomaly_stripes(matt_c, "Mattawamkeag — Flow Anomaly Stripes", axes[2])
plt.tight_layout()
save(fig, "anomaly_stripes")
plt.show()

---
## 24. Cumulative Flow — Water Year Accumulation

Cumulative monthly flow through each water year — steeper = wetter.
Comparing curves reveals years that started slow then caught up (late
snowmelt or fall storms) vs. years that were consistently dry.

In [ ]:
def plot_cumulative(obs, title, ax):
    years = sorted(obs.keys())
    cmap = plt.get_cmap("viridis")
    span = max(years) - min(years) or 1
    for y in years:
        c = cmap((y - min(years)) / span)
        cumul = np.cumsum(obs[y])
        ax.plot(range(1, 13), cumul, color=c, lw=0.6, alpha=0.5)
    # highlight median and latest
    stack = np.array([np.cumsum(obs[y]) for y in years])
    median_c = np.median(stack, axis=0)
    ax.plot(range(1, 13), median_c, color=AMBER, lw=2.5, label="median")
    latest = max(years)
    ax.plot(range(1, 13), np.cumsum(obs[latest]), color=CYAN, lw=2,
            label=f"{latest}")
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_ABBR, fontsize=7)
    ax.set_ylabel("cumulative flow (cfs·months)")
    ax.set_title(title)
    ax.legend(fontsize=7)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_cumulative(pen_c, "Penobscot — Cumulative", axes[0])
plot_cumulative(pisc_c, "Piscataquis — Cumulative", axes[1])
plot_cumulative(matt_c, "Mattawamkeag — Cumulative", axes[2])
plt.tight_layout()
save(fig, "cumulative")
plt.show()

---
## 25. Return Period Estimate — Annual Peaks

Flood frequency analysis using Weibull plotting position. Shows how
often a given peak flow is expected — the 100-year flood, the 10-year
flood. Critical for dam operations and floodplain management on the
Penobscot, which has major hydropower infrastructure.

In [ ]:
def plot_return_period(obs, title, ax):
    years = sorted(obs.keys())
    peaks = np.array([float(np.nanmax(obs[y])) for y in years])
    n = len(peaks)
    sorted_peaks = np.sort(peaks)[::-1]
    rank = np.arange(1, n + 1)
    return_period = (n + 1) / rank
    ax.semilogy(sorted_peaks, return_period, "o-", color=CYAN, ms=4, lw=1.2)
    for i in range(min(5, n)):
        yr_idx = int(np.where(peaks == sorted_peaks[i])[0][0])
        ax.annotate(str(years[yr_idx]), (sorted_peaks[i], return_period[i]),
                    fontsize=7, color=AMBER, xytext=(5, 5),
                    textcoords="offset points")
    ax.set_xlabel("peak monthly flow (cfs)")
    ax.set_ylabel("return period (years)")
    ax.set_title(title)
    ax.grid(True, alpha=0.2)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_return_period(pen_c, "Penobscot — Return Period", axes[0])
plot_return_period(pisc_c, "Piscataquis — Return Period", axes[1])
plot_return_period(matt_c, "Mattawamkeag — Return Period", axes[2])
plt.tight_layout()
save(fig, "return_period")
plt.show()

---
## 26. Lag-1 Autocorrelogram — Year Memory

Does this year's flow predict next year's? The autocorrelation function
measures persistence — significant lag-1 means wet years tend to follow
wet years (groundwater carry-over or climate persistence).

In [ ]:
def plot_autocorrelation(obs, title, ax, *, max_lag=15):
    years = sorted(obs.keys())
    annual = np.array([float(np.nanmean(obs[y])) for y in years])
    n = len(annual)
    mean = annual.mean()
    var = annual.var()
    acf = []
    for lag in range(max_lag + 1):
        if lag >= n:
            break
        cov = np.mean((annual[:n-lag] - mean) * (annual[lag:] - mean)) if lag > 0 else var
        acf.append(cov / var if var > 0 else 0)
    lags = range(len(acf))
    ax.bar(lags, acf, color=CYAN, alpha=0.7, width=0.8)
    ci = 1.96 / np.sqrt(n)
    ax.axhline(ci, color=MAGENTA, ls="--", lw=1, alpha=0.6)
    ax.axhline(-ci, color=MAGENTA, ls="--", lw=1, alpha=0.6)
    ax.axhline(0, color=MUTED, lw=0.5)
    ax.set_xlabel("lag (years)")
    ax.set_ylabel("autocorrelation")
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_autocorrelation(pen_c, "Penobscot — ACF", axes[0])
plot_autocorrelation(pisc_c, "Piscataquis — ACF", axes[1])
plot_autocorrelation(matt_c, "Mattawamkeag — ACF", axes[2])
plt.tight_layout()
save(fig, "autocorrelation")
plt.show()

---
## 27. Percentile Rank Forecast Context

Where does the most recent year sit in the historical distribution?
Each month's observed flow ranked against all prior years for that month.
Green = above median, red = below.

In [ ]:
def plot_percentile_context(obs, title, ax):
    years = sorted(obs.keys())
    latest = years[-1]
    stack = np.array([obs[y] for y in years])
    pctiles = []
    for m in range(12):
        col = stack[:, m]
        pct = np.sum(col <= obs[latest][m]) / len(col) * 100
        pctiles.append(pct)
    colors = [LIME if p >= 50 else RED for p in pctiles]
    ax.bar(range(1, 13), pctiles, color=colors, alpha=0.8, width=0.7)
    ax.axhline(50, color=MUTED, ls="--", lw=1)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_ABBR, fontsize=7)
    ax.set_ylabel("percentile rank")
    ax.set_ylim(0, 100)
    ax.set_title(f"{title} ({latest})")

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_percentile_context(pen_c, "Penobscot — Percentile", axes[0])
plot_percentile_context(pisc_c, "Piscataquis — Percentile", axes[1])
plot_percentile_context(matt_c, "Mattawamkeag — Percentile", axes[2])
plt.tight_layout()
save(fig, "percentile_context")
plt.show()

---
## 28. Cross-Gauge Correlation

How tightly do the three gauges track each other? High correlation
confirms they share a common climate driver; divergence reveals
local effects (land use, dam operations, tributary-specific storms).

In [ ]:
# overlap period — all three rivers
all_years = sorted(set(pen_c) & set(pisc_c) & set(matt_c))
print(f"Overlap period: {all_years[0]}–{all_years[-1]} ({len(all_years)} years)")

pen_ann = {y: float(np.mean(pen_c[y])) for y in all_years}
pisc_ann = {y: float(np.mean(pisc_c[y])) for y in all_years}
matt_ann = {y: float(np.mean(matt_c[y])) for y in all_years}

pairs = [
    ("Penobscot × Piscataquis", pen_ann, pisc_ann),
    ("Penobscot × Mattawamkeag", pen_ann, matt_ann),
    ("Piscataquis × Mattawamkeag", pisc_ann, matt_ann),
]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (label, a, b) in zip(axes, pairs):
    xs = [a[y] for y in all_years]
    ys = [b[y] for y in all_years]
    r = fm.correlate(a, b)
    ax.scatter(xs, ys, color=CYAN, s=15, alpha=0.7)
    z = np.polyfit(xs, ys, 1)
    xr = np.linspace(min(xs), max(xs), 50)
    ax.plot(xr, np.polyval(z, xr), "--", color=MAGENTA, lw=1.5)
    ax.set_xlabel(label.split(" × ")[0] + " (cfs)")
    ax.set_ylabel(label.split(" × ")[1] + " (cfs)")
    ax.set_title(f"{label}\nr = {r:+.3f}")
plt.tight_layout()
save(fig, "cross_gauge")
plt.show()

---
## 29. Forecast: Naive Climatological Projection

What does the historical record predict for next year? The simplest
forecast: mean monthly hydrograph with P10–P90 confidence bounds.
Not a real forecast — just the climatological envelope.

In [ ]:
def plot_climatological_forecast(obs, title, ax):
    years = sorted(obs.keys())
    latest = years[-1]
    stack = np.array([obs[y] for y in years])
    mean = np.nanmean(stack, axis=0)
    p10 = np.nanpercentile(stack, 10, axis=0)
    p90 = np.nanpercentile(stack, 90, axis=0)
    m = range(1, 13)
    ax.fill_between(m, p10, p90, color=CYAN, alpha=0.1, label="P10–P90")
    ax.plot(m, mean, color=CYAN, lw=2, label="climatological mean")
    ax.plot(m, obs[latest], color=AMBER, lw=2, ls="--",
            label=f"{latest} actual")
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_ABBR, fontsize=7)
    ax.set_ylabel("flow (cfs)")
    ax.set_title(title)
    ax.legend(fontsize=7)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_climatological_forecast(pen_c, "Penobscot — Clim. Forecast", axes[0])
plot_climatological_forecast(pisc_c, "Piscataquis — Clim. Forecast", axes[1])
plot_climatological_forecast(matt_c, "Mattawamkeag — Clim. Forecast", axes[2])
plt.tight_layout()
save(fig, "clim_forecast")
plt.show()

---
## 30. ENSO-Conditioned Forecast

If we know this winter's ENSO state, can we narrow the forecast?
The composite hydrograph for the current ENSO phase gives a
slightly more informed central estimate than raw climatology.

In [ ]:
def plot_enso_forecast(obs, oni_idx, title, ax):
    clean = complete_years(obs)
    latest_oni_year = max(y for y in oni_idx if y in clean)
    latest_oni = oni_idx[latest_oni_year]
    phase = "warm" if latest_oni > 0.5 else "cool" if latest_oni < -0.5 else "neutral"
    pc = fm.composite_hydrographs(clean, oni_idx)
    forecast = {"warm": pc.warm, "neutral": pc.neutral, "cool": pc.cool}[phase]
    stack = np.array([clean[y] for y in sorted(clean)])
    p10 = np.nanpercentile(stack, 10, axis=0)
    p90 = np.nanpercentile(stack, 90, axis=0)
    m = range(1, 13)
    ax.fill_between(m, p10, p90, color=MUTED, alpha=0.1, label="P10–P90 (all)")
    ax.plot(m, np.nanmean(stack, axis=0), color=MUTED, lw=1, ls="--", label="climatology")
    color = RED if phase == "warm" else COOL_BLUE if phase == "cool" else AMBER
    ax.plot(m, forecast, color=color, lw=2.5,
            label=f"ENSO {phase} composite")
    ax.plot(m, clean[latest_oni_year], color=CYAN, lw=1.5, ls=":",
            label=f"{latest_oni_year} actual")
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_ABBR, fontsize=7)
    ax.set_ylabel("flow (cfs)")
    ax.set_title(f"{title}\n{latest_oni_year} ONI = {latest_oni:+.2f} ({phase})")
    ax.legend(fontsize=7)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plot_enso_forecast(pen_c, oni, "Penobscot — ENSO Forecast", axes[0])
plot_enso_forecast(pisc_c, oni, "Piscataquis — ENSO Forecast", axes[1])
plot_enso_forecast(matt_c, oni, "Mattawamkeag — ENSO Forecast", axes[2])
plt.tight_layout()
save(fig, "enso_forecast")
plt.show()

---
## 31. Multi-Metric Dashboard

Key metrics at a glance for all three gauges — a one-table summary
of the Penobscot basin's hydrologic character.

In [ ]:
def compute_metrics(obs):
    clean = complete_years(obs)
    years = sorted(clean.keys())
    annual = [float(np.mean(clean[y])) for y in years]
    peaks = [float(np.max(clean[y])) for y in years]
    lows = [float(np.min(clean[y][5:8])) for y in years]
    flash = [fm.flashiness(clean[y]) for y in years]
    mean_flow = np.mean([clean[y] for y in years], axis=0)
    cot = fm.center_of_timing(mean_flow)
    sr = np.mean([fm.seasonal_ratio(clean[y]) for y in years])
    return {
        "Mean Flow": f"{np.mean(annual):,.0f} cfs",
        "Mean Peak": f"{np.mean(peaks):,.0f} cfs",
        "Mean Low": f"{np.mean(lows):,.0f} cfs",
        "Center of Timing": f"{cot:.1f}",
        "Seasonal Ratio": f"{sr:.2f}",
        "Mean Flashiness": f"{np.mean(flash):.4f}",
        "Record": f"{years[0]}–{years[-1]}",
        "N Years": len(years),
    }

print(f"{'Metric':<22} {'Penobscot':>14} {'Piscataquis':>14} {'Mattawamkeag':>14}")
print("─" * 66)
mp = compute_metrics(pen_c)
mpi = compute_metrics(pisc_c)
mm = compute_metrics(matt_c)
for k in mp:
    print(f"{k:<22} {str(mp[k]):>14} {str(mpi[k]):>14} {str(mm[k]):>14}")

---
## 32. Summary Statistics

Full statistical summary for each gauge — trend tests, teleconnections,
timing drift, and regime classification.

In [ ]:
def full_summary(obs, name):
    clean = complete_years(obs)
    years = sorted(clean.keys())
    annual = np.array([float(np.mean(clean[y])) for y in years])
    peaks = np.array([float(np.max(clean[y])) for y in years])
    lows = np.array([float(np.min(clean[y][5:8])) for y in years])
    mean_hydro = np.mean([clean[y] for y in years], axis=0)

    mk_ann = fm.mann_kendall(annual)
    mk_peak = fm.mann_kendall(peaks)
    mk_low = fm.mann_kendall(lows)
    tt = fm.center_of_timing_trend(clean)

    oni_r = fm.correlate({y: float(np.max(clean[y])) for y in years}, oni)
    pdo_r = fm.correlate({y: float(np.max(clean[y])) for y in years}, pdo)

    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"  Record:              {years[0]}–{years[-1]} ({len(years)} complete years)")
    print(f"  Mean annual flow:    {np.mean(annual):,.1f} cfs")
    print(f"  Std annual flow:     {np.std(annual):,.1f} cfs")
    print(f"  Mean peak monthly:   {np.mean(peaks):,.1f} cfs")
    print(f"  Mean summer low:     {np.mean(lows):,.1f} cfs")
    print(f"  Center of timing:    {fm.center_of_timing(mean_hydro):.2f} (month)")
    print(f"  Mean flashiness:     {np.mean([fm.flashiness(clean[y]) for y in years]):.4f}")
    print(f"  Mean seasonal ratio: {np.mean([fm.seasonal_ratio(clean[y]) for y in years]):.2f}")
    print(f"  ──── Trends ────")
    print(f"  Annual mean trend:   {mk_ann.trend} (τ={mk_ann.tau:+.3f}, p={mk_ann.p:.4f})")
    print(f"  Peak flow trend:     {mk_peak.trend} (τ={mk_peak.tau:+.3f}, p={mk_peak.p:.4f})")
    print(f"  Summer low trend:    {mk_low.trend} (τ={mk_low.tau:+.3f}, p={mk_low.p:.4f})")
    print(f"  Timing drift:        {tt.days_per_decade:+.1f} days/decade ({tt.trend})")
    print(f"  Annual Sen's slope:  {fm.sens_slope(annual):+.2f} cfs/yr")
    print(f"  ──── Teleconnections ────")
    print(f"  Peak × ONI (ENSO):   r = {oni_r:+.3f}")
    print(f"  Peak × PDO:          r = {pdo_r:+.3f}")

full_summary(pen_c, "Penobscot River at West Enfield, ME (01034500)")
full_summary(pisc_c, "Piscataquis River near Dover-Foxcroft, ME (01031500)")
full_summary(matt_c, "Mattawamkeag River near Mattawamkeag, ME (01030500)")

---

## Report Complete

**Penobscot River Basin, Maine** — 32 analytics panels across 3 gauges,
covering 100+ years of observed discharge. Key findings:

- **Regime:** Classic boreal snowmelt — sharp April freshet, low summer baseflow
- **Timing drift:** Watch for earlier spring runoff under warming (center-of-timing trend)
- **ENSO influence:** Weaker than Pacific NW but still detectable — La Niña winters
  tend to deliver more snowpack and a bigger spring flood
- **Century-scale:** The 100+ year Penobscot record is one of the longest continuous
  gauge records in New England — invaluable for climate-change detection

All data: USGS NWIS (federal public domain). All figures saved to `notebooks/figures/penobscot_*.png`.